# AION Clinical — Notebook-Demo

Ein vollständiger Workflow von Schema-Laden über Pattern-Mining bis kausaler Inferenz, alles in einem Jupyter-Notebook. Voraussetzung:

```bash
pip install -e ".[notebook]"
```

## 1. Schema laden

In [ ]:
from aion import TypeHierarchy
from aion.notebook import display_hierarchy

h = TypeHierarchy.from_yaml('../schemas/clinical_base.yaml')
display_hierarchy(h)

## 2. Synthetische Patientendaten erzeugen

In [ ]:
import random
from datetime import datetime, timedelta
from aion import ClinicalEvent, EventRelation, SQLiteEventStore
from aion.notebook import display_events

rng = random.Random(42)
store = SQLiteEventStore(':memory:')

base_phases = [
    ['Aufnahme', 'Fieber', 'Laktatmessung', 'Sepsis', 'Antibiotikum'],
    ['Aufnahme', 'Fieber', 'Antibiotikum'],
    ['Aufnahme', 'Beobachtung'],
    ['Aufnahme', 'Laktatmessung', 'Sepsis', 'Antibiotikum'],
]

for i in range(50):
    base = datetime(2026, 4, 1) + timedelta(days=i)
    stay = (base, base + timedelta(days=5))
    phase = rng.choice(base_phases)
    for offset, typ in enumerate(phase):
        store.add(ClinicalEvent(
            patient_id=f'P-{i:03d}',
            event_type=typ,
            t_start=base + timedelta(hours=offset),
            t_end=base + timedelta(hours=offset),
            stay_start=stay[0], stay_end=stay[1],
        ))

print(f'{store.count()} Events von 50 Patienten')
display_events(store.find_by_patient('P-007'))

## 3. Pattern-Mining auf den Sequenzen

In [ ]:
from aion import TCFG
from aion.notebook import sequences_from_store, plot_pattern_support

sequences = sequences_from_store(store)
patterns = TCFG.mine_patterns(
    sequences,
    min_length=2, max_length=5,
    min_support=0.30,
)
print(f'{len(patterns)} frequente Phasen gefunden')

fig = plot_pattern_support(patterns, top=10, title='Top-10 frequente Klinik-Phasen')
fig

## 4. Kausalgraph visualisieren

In [ ]:
from aion import CausalGraph
from aion.notebook import plot_causal_graph

g = CausalGraph()
g.add_edge('SOFA', 'FrueheAntibiose')
g.add_edge('SOFA', 'Mortalitaet')
g.add_edge('FrueheAntibiose', 'Mortalitaet')
g.add_edge('FrueheAntibiose', 'ICUTage')
g.add_edge('ICUTage', 'Mortalitaet')

bs = g.find_backdoor_adjustment_set('FrueheAntibiose', 'Mortalitaet')
print(f'Backdoor-Adjustment-Set: {bs}')

fig = plot_causal_graph(
    g,
    treatment='FrueheAntibiose',
    outcome='Mortalitaet',
    backdoor_set=bs,
    title='Kausalmodell: hilft frühe Antibiose?',
)
fig

## 5. Backdoor-Validierung formell

In [ ]:
from aion import is_valid_backdoor_set

report = is_valid_backdoor_set(g, 'FrueheAntibiose', 'Mortalitaet', bs)
print(report)

## Schluss

Die fünf AION-Operationen aus diesem Notebook (Schema laden, Daten erzeugen, Pattern-Mining, Kausalgraph, Backdoor-Validierung) decken den typischen Statistik-Workflow ab. Erweiterungen:

- **Echte FHIR-Daten:** `aion.fhir.from_fhir_bundle(bundle)`
- **Sensitivitätsanalyse:** `aion.dowhy.refute_estimate(...)`
- **CLI-Pipeline:** `!aion mine my.db --min-support 0.3` (in einer Notebook-Zelle)